# CRISP-DM Fase 1: Business Understanding (Comprensión del Negocio)
## Dataset: IDEAM Climatología e Índices Hidroclimáticos Históricos

**Ecosistema**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 1: Business Understanding) | **Fase PDCO**: PLAN  
**Estándares Aplicados**: SWEBOK Cap. 1 (Requerimientos de Software), DAMA-DMBOK 2 (Metadata & Governance), ISO/IEC 25010.

---

### 1. Contexto Estratégico y Problema de Negocio
El balance hídrico (P - ETc) determina la aptitud de los suelos agrícolas. Las precipitaciones presentan regímenes bi-modales o mono-modales que condicionan la fenología de los cultivos.

**Objetivo de Negocio**:
> Determinar la oferta hídrica histórica y los regímenes de lluvia para correlacionar los ciclos de siembra y cosecha con la variabilidad climática territorial.

### 2. Entidad del Dominio y Alcance
* **Entidad Analítica Principal**: `Series Hidroclimatológicas de Precipitación, Evapotranspiración y Temperatura`
* **Custodio Oficial de los Datos**: Instituto de Hidrología, Meteorología y Estudios Ambientales (IDEAM)
* **Granularidad Espacial**: `Puntual: Estación Meteorológica (Latitud, Longitud, Altitud msnm)`
* **Granularidad Temporal**: `Diaria / Mensual consolidada`

### 3. Preguntas de Negocio Mapeadas (`docs/1-Bateria_preguntas.md`):
* **B3: Estabilidad Climática**
* **F1: Estacionalidad Pluviométrica**
* **E3: Balance Hídrico Territorial (BH = P - ETc)**

### 4. Indicadores Clave de Desempeño (KPIs):
* `Precipitación acumulada mensual (mm)`
* `Índice de anomalía de lluvia (SPI)`
* `Balance hídrico neto`

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# Este bloque garantiza la disponibilidad de librerías en cualquier entorno cloud.
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic missingno


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN DE RUTAS DEL PROYECTO (PEP 8)]
# Importación estándar y resolución dinámica del path para src/
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolver raíz del proyecto para importar módulos de src/
CURRENT_DIR = Path(".").resolve()
APP_ROOT = CURRENT_DIR.parent.parent
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno inicializado exitosamente. Raíz del proyecto: {APP_ROOT}")


### 5. Fundamentación del Contrato de Datos (Data Contract)
Bajo los lineamientos de **DAMA-DMBOK 2**, todo pipeline de datos analíticos debe iniciar con la verificación de un **Contrato de Datos**. 
El contrato define formalmente:
1. El custodio del dato y la periodicidad de actualización.
2. La definición operativa de las variables críticas y sus tipos de datos esperados.
3. Las tolerancias de calidad (porcentaje máximo de nulos y umbrales de detección de valores anómalos).


In [ ]:
# ==============================================================================
# [FASE 1: VERIFICACIÓN DEL CONTRATO DE NEGOCIO Y ESPECIFICACIÓN TÉCNICA]
# ==============================================================================
import yaml

config_path = APP_ROOT / "config" / "datasets_config.yaml"
if config_path.exists():
    with open(config_path, "r", encoding="utf-8") as f:
        config_data = yaml.safe_load(f)
    print(f"📄 Archivo de configuración cargado: {config_path.name}")
    dataset_cfg = config_data.get("datasets", {}).get("05_ideam_climatologia", {})
    print(f"📋 Especificación del Dataset en YAML: {dataset_cfg}")
else:
    print("ℹ️ Configuración YAML por defecto activa.")

print("\n--- DEFINICIÓN FORMAL DEL CONTRATO ---")
print("• Entidad:", "Series Hidroclimatológicas de Precipitación, Evapotranspiración y Temperatura")
print("• Custodio:", "Instituto de Hidrología, Meteorología y Estudios Ambientales (IDEAM)")
print("• Granularidad Espacial:", "Puntual: Estación Meteorológica (Latitud, Longitud, Altitud msnm)")
print("• Granularidad Temporal:", "Diaria / Mensual consolidada")
print("• KPIs de Negocio:", ['Precipitación acumulada mensual (mm)', 'Índice de anomalía de lluvia (SPI)', 'Balance hídrico neto'])


### 6. Verificación de Disponibilidad en el Data Lakehouse
El Lakehouse implementa la arquitectura Medallion:
* **Bronze**: Datos crudos inmutables preservando el formato original del custodio.
* **Silver**: Datos limpios, sanitizados y estructurados en formato columnar Parquet comprimido con Snappy.
* **Gold**: Base de datos relacional SQLite (`agrostats_lakehouse.db`) con esquemas conformados e índices para consultas analíticas de alta velocidad.


In [ ]:
# ==============================================================================
# [FASE 1: INSPECCIÓN DE DISPONIBILIDAD EN LAS CAPAS DEL LAKEHOUSE]
# ==============================================================================
from src.database.db_manager import DatabaseManager

db_path = APP_ROOT / "data" / "processed" / "agrostats_lakehouse.db"
parquet_path = APP_ROOT / "data/processed/ideam_pluviometria.parquet"

print("🔍 Verificación de Almacenamiento en Data Lakehouse:")
print(f"• Capa Silver (Parquet): {parquet_path.name} -> Existe: {parquet_path.exists()}")
print(f"• Capa Gold (SQLite DB): {db_path.name} -> Existe: {db_path.exists()}")

if parquet_path.exists():
    df_sample = pd.read_parquet(parquet_path)
    print(f"\n✅ Contrato de datos preliminar verificado exitosamente:")
    print(f"• Total de registros curados: {len(df_sample):,} filas")
    print(f"• Columnas disponibles: {list(df_sample.columns)}")


### 7. Síntesis y Transición a la Fase 2 (Data Understanding)
* **Entregable de la Fase 1**: Especificación formal del problema de negocio, KPIs concertados y contrato de datos validado.
* **Siguiente Paso**: Proceder al notebook `02_ingestion.ipynb` para ejecutar la extracción inmutable y auditoría criptográfica SHA-256.
